<a href="https://colab.research.google.com/github/MedPhysTools/3D-Gamma/blob/main/ICTP_brain_tumor_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Brain-tumor detection lab — YOLO26 + FROC

A **self-contained** Colab: install Ultralytics, train YOLO26 on the brain-tumor dataset, then play with detection thresholds — see a **collage of ground-truth vs detected tumors** update as you move the sliders — while each run logs one **FROC point** (sensitivity vs false positives per image) to a shared Google Sheet.

Run top to bottom: **`INSTALL` → `TRAIN` → `EXPERIMENT` → `DASHBOARD`**.

**Why FROC (not ROC) for detection.** A ROC needs a False Positive Rate, `FP / (FP + TN)`; in object detection the number of true negatives (background boxes) is undefined, so ROC doesn't apply. FROC keeps sensitivity on the y-axis and uses **false positives per image** on the x-axis. A *positive* prediction is a true positive when it overlaps a ground-truth tumor box with **IoU ≥ `iou_thr`**; unmatched positive predictions are false positives.

**Two knobs to play with.** `conf` (how confident a detection must be) and `iou_thr` (how strict the overlap must be to count as a hit). The collage shows the same fixed images every run, so their effect is visible; each run also logs a point, so the class curve fills up as everyone plays.

Sheet: share as **Anyone with the link → Editor** (already wired in); header is created automatically. **Reusing an earlier sheet? Clear it first** (select all cells → Delete) — this version adds an `iou_thr` column. *Class-scale tip:* train once, share `best.pt`, students set `weights_path` and skip `TRAIN` (the experiment is inference-only).

```text
    _____   ________________    __    __
   /  _/ | / / ___/_  __/   |  / /   / /
   / //  |/ /\__ \ / / / /| | / /   / /  
 _/ // /|  /___/ // / / ___ |/ /___/ /___
/___/_/ |_//____//_/ /_/  |_/_____/_____/
                                         
```

Install Ultralytics (YOLO26) and print the environment / GPU check.

In [ ]:
#@title ⚙ Install Ultralytics and run environment checks
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

```text
  __________  ___    _____   __
 /_  __/ __ \/   |  /  _/ | / /
  / / / /_/ / /| |  / //  |/ /
 / / / _, _/ ___ |_/ // /|  /  
/_/ /_/ |_/_/  |_/___/_/ |_/   
                               
```

Train YOLO26-nano on the brain-tumor dataset (auto-downloaded). **Skip if you were given a shared `best.pt`** — set `weights_path` in the EXPERIMENT cell instead.

In [ ]:
#@title 🚆 Train YOLO26 on the brain-tumor dataset
epochs = 25 #@param {type:"slider", min:5, max:60, step:5}
imgsz  = 640 #@param [320, 512, 640] {type:"raw"}

from ultralytics import YOLO
model = YOLO("yolo26n.pt")                       # COCO-pretrained nano detector
results = model.train(data="brain-tumor.yaml", epochs=epochs, imgsz=imgsz)

```text
    _______  __ ____  __________  ______  __________   ________
   / ____/ |/ // __ \/ ____/ __ \/  _/  |/  / ____/ | / /_  __/
  / __/  |   // /_/ / __/ / /_/ // // /|_/ / __/ /  |/ / / /   
 / /___ /   |/ ____/ /___/ _, _// // /  / / /___/ /|  / / /    
/_____//_/|_/_/   /_____/_/ |_/___/_/  /_/_____/_/ |_/ /_/     
                                                               
```

**Play with the sliders next to the images.** `conf` and `iou_thr` update the collage live — green = ground truth, blue = detected (TP), red = false positive. The side panel shows the current **sensitivity** and **FP/image** over the whole validation set. Type your name and click **Log this point** to add it to the class curve.

The validation set is scored **once** when the cell starts (a few seconds); after that the sliders are instant, because they only filter and re-match the cached detections — no new inference.

Few false positives? Lower `conf` (0.05–0.10) or raise `iou_thr`, and use the **show** selector to focus the collage on false-positive or tumour-absent images.

In [ ]:
#@title ▶ Interactive collage + FROC logger (sliders beside the images)
weights_path = "runs/detect/train/weights/best.pt"   # used only if no trained model is in memory

import os, time, datetime, math, random, numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt, matplotlib.patches as patches
from matplotlib.offsetbox import TextArea, HPacker, AnnotationBbox
import ipywidgets as W
from IPython.display import display, clear_output
from ultralytics import YOLO

try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

if "model" not in globals():
    if not os.path.exists(weights_path):
        raise FileNotFoundError("No `model` in memory and no weights at '" + weights_path +
                                "'. Run TRAIN first, or upload a best.pt and set weights_path.")
    model = YOLO(weights_path)

def find_val():
    cands = []
    try:
        from ultralytics.utils import SETTINGS
        cands.append(Path(SETTINGS.get("datasets_dir", ".")) / "brain-tumor")
    except Exception:
        pass
    cands += [Path("datasets/brain-tumor"), Path("/content/datasets/brain-tumor"),
              Path.home() / "datasets/brain-tumor"]
    for c in cands:
        if (c / "images/val").exists():
            return c / "images/val", c / "labels/val"
    raise FileNotFoundError("brain-tumor val split not found - run TRAIN first.")
val_img, val_lbl = find_val()
imgs = sorted(p for p in val_img.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
POS = 1

def gt_pos_boxes(p, w, h):
    boxes = []
    lp = val_lbl / (p.stem + ".txt")
    if lp.exists():
        for line in lp.read_text().splitlines():
            t = line.split()
            if len(t) >= 5 and int(float(t[0])) == POS:
                cx, cy, bw, bh = map(float, t[1:5])
                boxes.append([(cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h])
    return np.array(boxes, dtype=float) if boxes else np.zeros((0, 4))

def iou_xyxy(a, b):
    x1 = np.maximum(a[0], b[:, 0]); y1 = np.maximum(a[1], b[:, 1])
    x2 = np.minimum(a[2], b[:, 2]); y2 = np.minimum(a[3], b[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[2]-a[0]) * (a[3]-a[1]); ab = (b[:, 2]-b[:, 0]) * (b[:, 3]-b[:, 1])
    return inter / (aa + ab - inter + 1e-9)

# ---- score the whole val set ONCE at low conf, then cache ----
print("Scoring the validation set once (a few seconds)...")
cache = []
preds = model.predict([str(p) for p in imgs], conf=0.001, verbose=False, stream=True)
for p, r in zip(imgs, preds):
    h, w = r.orig_shape
    gts = gt_pos_boxes(p, w, h)
    if r.boxes is not None and len(r.boxes):
        cls = r.boxes.cls.cpu().numpy(); cf = r.boxes.conf.cpu().numpy(); xy = r.boxes.xyxy.cpu().numpy()
        m = cls == POS; pboxes = xy[m]; pconf = cf[m]
    else:
        pboxes = np.zeros((0, 4)); pconf = np.zeros((0,))
    cache.append({"path": str(p), "gts": gts, "pboxes": pboxes, "pconf": pconf})
print("Ready - move the sliders.")

pos_idx = [i for i, c in enumerate(cache) if len(c["gts"]) > 0]
neg_idx = [i for i, c in enumerate(cache) if len(c["gts"]) == 0]
random.Random(0).shuffle(pos_idx); random.Random(1).shuffle(neg_idx)

def match(gts, pboxes, pconf, conf, iou_thr):
    keep = pconf >= conf
    pb = pboxes[keep]; pc = pconf[keep]
    pb = pb[np.argsort(-pc)]
    matched = np.zeros(len(gts), dtype=bool); flags = []; tp = fp = 0
    for box in pb:
        is_tp = False
        if len(gts):
            ious = iou_xyxy(box, gts); j = int(np.argmax(ious))
            if ious[j] >= iou_thr and not matched[j]:
                matched[j] = True; is_tp = True
        flags.append(is_tp); tp += int(is_tp); fp += int(not is_tp)
    return tp, fp, pb, flags

def froc_point(conf, iou_thr):
    TP = FP = GT = 0
    for c in cache:
        tp, fp, _, _ = match(c["gts"], c["pboxes"], c["pconf"], conf, iou_thr)
        TP += tp; FP += fp; GT += len(c["gts"])
    return (TP / GT if GT else 0.0), FP / len(cache), TP, FP, GT

def pick_indices(focus, conf, iou_thr, n):
    if focus == "tumor present":
        return sorted(pos_idx[:n])
    if focus == "tumor absent":
        return sorted(neg_idx[:n]) if neg_idx else sorted(pos_idx[:n])
    # rank images by number of false positives at the current thresholds
    fpc = []
    for i, c in enumerate(cache):
        _, fp, _, _ = match(c["gts"], c["pboxes"], c["pconf"], conf, iou_thr)
        if fp > 0:
            fpc.append((fp, i))
    fpc.sort(reverse=True)
    fp_idx = [i for _, i in fpc]
    if focus == "false positives":
        return sorted(fp_idx[:n]) if fp_idx else sorted(pos_idx[:n])
    # mixed: half tumour cases, half worst false-positive cases
    half = max(1, n // 2)
    a = pos_idx[:half]
    b = [i for i in fp_idx if i not in a][:n - len(a)]
    return sorted(a + b) if (a or b) else sorted(pos_idx[:n])

# ---- widgets: controls on the left, collage on the right ----
name_w = W.Text(value="", description="name")
conf_s = W.FloatSlider(value=0.25, min=0.05, max=0.90, step=0.05, description="conf", continuous_update=False)
iou_s  = W.FloatSlider(value=0.50, min=0.10, max=0.70, step=0.05, description="IoU", continuous_update=False)
n_dd   = W.Dropdown(options=[4, 6, 9], value=6, description="images")
focus_dd = W.Dropdown(options=["tumor present", "false positives", "tumor absent", "mixed"], value="tumor present", description="show")
log_btn = W.Button(description="Log this point", button_style="success", icon="check")
status = W.HTML()
out = W.Output()

SHEET_URL = "https://docs.google.com/spreadsheets/d/1_-kntyt1fDtnypsKNcVb9zEVrtoevlwu3sPigNH5-hs/edit?usp=sharing"
HEADER = ["timestamp", "name", "conf", "iou_thr", "sensitivity", "fp_per_image", "n_tp", "n_fp"]

def colored_title(ax, gtn, tpn, fpn, fontsize=8, y=1.02):
    # multicolour title matching the box colours: GT green, TP blue, FP red
    parts = [("GT %d" % gtn, "lime"), ("  |  ", "0.5"),
             ("TP %d" % tpn, "deepskyblue"), ("  |  ", "0.5"),
             ("FP %d" % fpn, "red")]
    tbxs = [TextArea(t, textprops=dict(color=col, fontsize=fontsize, weight="bold")) for t, col in parts]
    ab = AnnotationBbox(HPacker(children=tbxs, align="center", pad=0, sep=0),
                        (0.5, y), xycoords="axes fraction", frameon=False, box_alignment=(0.5, 0))
    ax.add_artist(ab)

def render(*_):
    sens, fppi, TP, FP, GT = froc_point(conf_s.value, iou_s.value)
    status.value = ("<b>conf=%.2f&nbsp;&nbsp;IoU&ge;%.2f</b><br>sensitivity = <b>%.3f</b>"
                    "<br>FP / image = <b>%.3f</b><br><span style='color:gray'>TP=%d/%d&nbsp;FP=%d</span>"
                    % (conf_s.value, iou_s.value, sens, fppi, TP, GT, FP))
    idxs = pick_indices(focus_dd.value, conf_s.value, iou_s.value, n_dd.value)
    with out:
        clear_output(wait=True)
        ncol = 3; nrow = math.ceil(len(idxs) / ncol)
        fig, axes = plt.subplots(nrow, ncol, figsize=(3.3*ncol, 3.3*nrow))
        axes = np.array(axes).reshape(-1)
        for ax, i in zip(axes, idxs):
            c = cache[i]
            ax.imshow(Image.open(c["path"]).convert("RGB")); ax.axis("off")
            tp, fp, pb, flags = match(c["gts"], c["pboxes"], c["pconf"], conf_s.value, iou_s.value)
            for g in c["gts"]:
                ax.add_patch(patches.Rectangle((g[0], g[1]), g[2]-g[0], g[3]-g[1],
                                               fill=False, edgecolor="lime", lw=2))
            for box, tpf in zip(pb, flags):
                ax.add_patch(patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                                               fill=False, edgecolor=("deepskyblue" if tpf else "red"), lw=2))
            colored_title(ax, len(c["gts"]), tp, fp)
        for ax in axes[len(idxs):]:
            ax.axis("off")
        plt.tight_layout(); plt.show()

def on_log(_):
    try:
        from google.colab import auth; auth.authenticate_user()
        import gspread
        from google.auth import default
        creds, _ = default(); gc = gspread.authorize(creds)
        ws = gc.open_by_url(SHEET_URL).sheet1
        values = ws.get_all_values()
        if not values:
            ws.append_row(HEADER)
        elif values[0] != HEADER:
            status.value += "<br><span style='color:red'>Sheet header mismatch - clear the sheet and retry.</span>"
            return
        sens, fppi, TP, FP, GT = froc_point(conf_s.value, iou_s.value)
        row = [datetime.datetime.now().isoformat(timespec="seconds"), name_w.value or "anon",
               conf_s.value, iou_s.value, round(sens, 4), round(fppi, 4), TP, FP]
        ws.append_row(row, value_input_option="USER_ENTERED")
        status.value += "<br><span style='color:green'>Logged ✓ (conf=%.2f, IoU=%.2f)</span>" % (conf_s.value, iou_s.value)
    except Exception as e:
        status.value += "<br><span style='color:red'>Log failed: %s</span>" % e

for wdg in (conf_s, iou_s, n_dd, focus_dd):
    wdg.observe(render, "value")
log_btn.on_click(on_log)

controls = W.VBox([name_w, conf_s, iou_s, n_dd, focus_dd, log_btn, status])
display(W.HBox([controls, out]))
render()

```text
    ____  ___   _____ __  ______  ____  ___    ____  ____
   / __ \/   | / ___// / / / __ )/ __ \/   |  / __ \/ __ \
  / / / / /| | \__ \/ /_/ / __  / / / / /| | / /_/ / / / /
 / /_/ / ___ |___/ / __  / /_/ / /_/ / ___ |/ _, _/ /_/ /
/_____/_/  |_/____/_/ /_/_____/\____/_/  |_/_/ |_/_____/  
                                                          
```

**Project this and re-run as points come in.** Every logged run is one point in the cloud (colour = `iou_thr`); a single saturating curve `sensitivity = a·(1 − e^(−k·FP/img))` is fitted across the whole class. Sensitivity rises with the false positives allowed, then plateaus.

In [ ]:
#@title 📊 Class FROC: point cloud + single class fit
from google.colab import auth; auth.authenticate_user()
import gspread, numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.auth import default
creds, _ = default(); gc = gspread.authorize(creds)
SHEET_URL = "https://docs.google.com/spreadsheets/d/1_-kntyt1fDtnypsKNcVb9zEVrtoevlwu3sPigNH5-hs/edit?usp=sharing"
ws = gc.open_by_url(SHEET_URL).sheet1

try:
    records = ws.get_all_records(value_render_option="UNFORMATTED_VALUE")
except TypeError:
    records = ws.get_all_records()
df = pd.DataFrame(records)

need = {"conf", "sensitivity", "fp_per_image"}
if df.empty:
    print("No data yet - run the EXPERIMENT cell first.")
elif not need.issubset(df.columns):
    print("Unexpected columns in the sheet:", list(df.columns))
    print("This sheet holds data from an earlier version. Clear it (select all > Delete) and re-log.")
else:
    cols = ["conf", "sensitivity", "fp_per_image"] + (["iou_thr"] if "iou_thr" in df.columns else [])
    for c in cols:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    df = df.dropna(subset=["sensitivity", "fp_per_image"])
    if df.empty:
        print("Rows found but the numbers could not be parsed - check the sheet values.")
    else:
        x = df["fp_per_image"].to_numpy(); y = df["sensitivity"].to_numpy()
        plt.figure(figsize=(7, 5))

        # point cloud (no connecting lines); colour by IoU threshold if available
        if "iou_thr" in df.columns and df["iou_thr"].notna().any():
            sc = plt.scatter(x, y, c=df["iou_thr"], cmap="viridis", s=45, alpha=.8,
                             edgecolor="k", linewidth=.3)
            plt.colorbar(sc, label="IoU match threshold")
        else:
            plt.scatter(x, y, s=45, alpha=.8, edgecolor="k", linewidth=.3)

        # single saturating fit across ALL class points
        def sat(xx, a, k): return a * (1.0 - np.exp(-k * xx))
        if len(df) >= 3 and np.ptp(x) > 0:
            try:
                from scipy.optimize import curve_fit
                p0 = [min(1.0, max(float(y.max()), 0.5)), 1.0]
                popt, _ = curve_fit(sat, x, y, p0=p0, bounds=([0, 0], [1.0, np.inf]), maxfev=10000)
                xs = np.linspace(0, float(x.max()) * 1.05, 200)
                yhat = sat(x, *popt)
                ss_res = float(np.sum((y - yhat) ** 2)); ss_tot = float(np.sum((y - y.mean()) ** 2))
                r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
                plt.plot(xs, sat(xs, *popt), "-", color="crimson", lw=2,
                         label="class fit: a=%.2f, k=%.2f (R²=%.2f)" % (popt[0], popt[1], r2))
                plt.legend(loc="lower right")
            except Exception as e:
                print("Fit skipped:", e)
        else:
            print("Need at least 3 points spread over different FP/image values to fit a curve.")

        plt.xlabel("False positives per image"); plt.ylabel("Sensitivity (per lesion)")
        plt.ylim(0, 1.02); plt.xlim(left=0)
        plt.title("Class FROC - %d operating points" % len(df))
        plt.grid(alpha=.3); plt.show()